# Azure OpenAI Image Evaluation Demo

A demonstration of Azure OpenAI image evaluation workflow.

## Prerequisites
- Azure OpenAI account with image model deployed
- Azure CLI authenticated with access to the Azure AI project

In [1]:
from dotenv import load_dotenv
from scripts.eval_utils import AsyncEvalClient

# Load environment variables
load_dotenv()

# Setup evaluation client with Azure CLI authentication
PROJECT_ENDPOINT = "https://eastus2-prakharg-demo-2026.services.ai.azure.com/api/projects/eastus2-prakharg-demo-2026"
client = AsyncEvalClient(project_endpoint=PROJECT_ENDPOINT)
print("🎉 Azure OpenAI Evaluation Client ready!")

IMAGE_MODEL_DEPLOYMENT_NAME = "gpt-4o-mini"

🎉 Azure OpenAI Evaluation Client ready!


## Load the dataset and upload to your Azure OpenAI account
- This step loads a deterministic set of public JPEG images with literal reference captions
- Upload the prepared evaluation file to your Azure OpenAI account. This file will be used as the data source for running evaluations.

In [2]:
from scripts.image_utils import load_and_create_image_dataset, display_items

# Load dataset from Hugging Face and create the image dataset for evaluation
load_and_create_image_dataset("picsum", max_samples=10)

# Display the created evaluation file
display_items(3)


C:\Users\prakharg\.copilot\session-state\f1833604-4ea0-4404-893e-2421eb3ad912\files\evaluation-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Created evaluation file with 10 items
{
  "item": {
    "image_url": "https://fastly.picsum.photos/id/10/512/512.jpg?hmac=Xno3CU-pY4UWutaA1XRUThqgbV_vn7kN5PjrlkRw-9Q",
    "caption": "A view of a calm ocean with distant islands, framed by lush green trees."
  }
}
{
  "item": {
    "image_url": "https://fastly.picsum.photos/id/20/512/512.jpg?hmac=pGfehaSsk_8XvaGx8DJNw0BHdYqBj6CZWPFu6sTNPiA",
    "caption": "A workspace featuring a laptop, smartphone, books, and design materials."
  }
}
{
  "item": {
    "image_url": "https://fastly.picsum.photos/id/30/512/512.jpg?hmac=ph0RnBNUKGfVgtH1-jDWPbN0TZXOXat6xKzPazurl_Y",
    "caption": "A white mug featuring a design inspired by a Cuban stamp."
  }
}


In [3]:
# Upload the evaluation file to Azure OpenAI
eval_file_id = await client.upload_file(
    file_name="image_emotion_evaluation.jsonl",
    file_path="./data/image_emotion_evaluation.jsonl")
print(f"✅ Eval file ID: {eval_file_id}")

File 'image_emotion_evaluation.jsonl' already exists. Returning existing file ID.
✅ Eval file ID: file-44cf48a92be34e24af0663418e8828aa


## Create Evaluation with Score Model Grader

Define a score model grader that will evaluate how well the model's generated captions match the expected captions and accurately describe the images. The grader:
- Uses the same image model to judge the quality
- Scores responses from 0 (poor match) to 1 (excellent match)
- Sets a passing threshold of 0.5

In [4]:
score_model = {
      "type": "score_model",
      "name": "Image to Text Grader",
      "model": IMAGE_MODEL_DEPLOYMENT_NAME,
      "input": [
        {
          "role": "system",
          "content": "You are an expert grader. Judge how well the model response {{sample.output_text}} describes the image as well as matches the caption {{item.caption}}. Output a score of 1 if its an excelent match with both. If it's somewhat compatible, output a score around 0.5. Otherwise, give a score of 0."
        },
        {
            "role": "user",
            "content": [
                { 
                    "type": "input_text", 
                    "text": "Caption: {{ item.caption }}"
                },
                { 
                    "type": "input_image", 
                    "image_url": "{{ item.image_url }}"
                }
            ]
        }
      ],
      "range": [
        0,
        1
      ],
      "pass_threshold": 0.5
    }

eval_id = await client.create_eval_sdk(
    name="Image Caption Evaluation",
    testing_criteria=[score_model],
    data_source_config={
    "type": "custom",
    "item_schema": {
      "type": "object",
      "properties": {
        "image_url": {
          "type": "string",
          "description": "The URL of the image to be evaluated."
        },
        "caption": {
          "type": "string",
          "description": "The caption describing the image."
        }
      },
      "required": [
        "image_url",
        "caption"
      ]
    },
    "include_sample_schema": True,
  })

Evaluation created successfully with ID: eval_496ae605a6cf4eb180429c88b341be87


## Run the Evaluation

Generate captions with the vision deployment, attach them as `sample.output_text`, and submit a JSONL evaluation run. This defines:
- Which model generates the image captions
- The system and user prompts used for caption generation
- The image message format used by chat completions
- The supplied samples evaluated by the score-model grader

In [5]:
import json

with open("./data/image_emotion_evaluation.jsonl", encoding="utf-8") as f:
    eval_items = [json.loads(line) for line in f]

for eval_item in eval_items:
    item = eval_item["item"]
    response = await client.client.chat.completions.create(
        model=IMAGE_MODEL_DEPLOYMENT_NAME,
        messages=[
            {
                "role": "system",
                "content": "Describe the image accurately and concisely."
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Write one short caption."},
                    {
                        "type": "image_url",
                        "image_url": {"url": item["image_url"], "detail": "auto"}
                    }
                ]
            }
        ],
        max_tokens=100,
        temperature=0
    )
    eval_item["sample"] = {"output_text": response.choices[0].message.content}

data_source = {
    "type": "jsonl",
    "source": {
      "type": "file_content",
      "content": eval_items
    }
}

run = await client.create_eval_run_sdk(eval_id, "Image Caption Evaluation", data_source)
run_id = run['id']


Created evaluation run for Image Caption Evaluation: evalrun_ddadbc810a324f2c86ea8f5f3ef20c99


## Monitor and Display Results

Poll the evaluation run status until completion, then retrieve and display the results in a DataFrame showing:
- Item IDs
- Grading results (pass/fail status)
- Expected captions from the dataset
- Model-generated responses for comparison

In [6]:
import asyncio
import pandas as pd

while True:
    run = await client.get_eval_run_sdk(eval_id=eval_id, run_id=run_id)
    if run['status'] == "completed":
        output_items_response = await client.get_eval_run_output_items_sdk(
            eval_id=eval_id, run_id=run_id)

        # Get the actual list of items from the response object
        output_items = output_items_response.data if hasattr(output_items_response, 'data') else output_items_response

        # Create DataFrame with safe access to nested fields
        df_data = {
            "id": [],
            "grading_results": [],
            "expected_caption": [],
            "model_response": []
        }

        for item in output_items:
            # Convert Pydantic model to dict if needed
            item_dict = item.model_dump() if hasattr(item, 'model_dump') else item
            
            df_data["id"].append(item_dict.get("id", "N/A"))
            results = item_dict.get("results") or []
            result = results[0] if results else {}
            df_data["grading_results"].append({
                "status": result.get("status", "N/A"),
                "score": result.get("score")
            })

            # Safely get expected caption
            datasource_item = item_dict.get('datasource_item', {})
            df_data["expected_caption"].append(datasource_item.get("caption", "N/A"))
            
            # Extract the generated caption when available
            provided_output = datasource_item.get("sample.output_text")
            sample = item_dict.get("sample", {})
            output = sample.get("output") or []
            output_transcript = provided_output or (output[0].get("content", "N/A") if output else "N/A")
            df_data["model_response"].append(output_transcript)

        df = pd.DataFrame(df_data)
        display(df)
        break
    if run['status'] == "failed":
        print("Evaluation run failed:")
        raise RuntimeError(run.get('error', 'Unknown error'))
    print(f"Status: {run['status']}. Waiting...")
    await asyncio.sleep(5)


Status: in_progress. Waiting...


Status: in_progress. Waiting...


Status: in_progress. Waiting...


Status: in_progress. Waiting...


Status: in_progress. Waiting...


...

Status: in_progress. Waiting...


Status: in_progress. Waiting...


Status: in_progress. Waiting...


Status: in_progress. Waiting...


,id,grading_results,expected_caption,model_response
0,1,"{'status': 'completed', 'score': 1.0}","A view of a calm ocean with distant islands, f...","""Serenity at the edge of the forest, where lan..."
1,2,"{'status': 'completed', 'score': 1.0}","A workspace featuring a laptop, smartphone, bo...","""Creative workspace: where ideas come to life."""
2,3,"{'status': 'completed', 'score': 1.0}",A white mug featuring a design inspired by a C...,"""Embrace the warmth of nostalgia with this vin..."
3,4,"{'status': 'completed', 'score': 0.5}",Close-up of a rabbit's nose and whiskers.,"""Up close and personal with a furry friend!"""
4,5,"{'status': 'completed', 'score': 1.0}",A bird spreads its wings on a rope near a boat...,"""Serenity by the water: a bird drying its wing..."
5,6,"{'status': 'completed', 'score': 1.0}",A top-down view of a workspace with a computer...,"""An organized workspace ready for creativity a..."
6,7,"{'status': 'completed', 'score': 1.0}",A foggy road lined with tall trees.,"A serene road lined with trees, shrouded in mo..."
7,8,"{'status': 'completed', 'score': 0.5}",A black-and-white close-up of pine cones on wood.,"""Nature's intricate beauty captured in monochr..."
8,9,"{'status': 'completed', 'score': 1.0}",A row of glass jars placed on wooden stakes in...,"""Whimsical garden decor: glass jars perched on..."
